# Module 5.3: Data Quality & Validation

**Level:** 2 | **Duration:** 35 minutes | **Prerequisites:** M5.1, M5.2, L1 M1.3

## Overview

Production RAG systems often index 30-40% low-quality or duplicate content, wasting storage, compute resources, and degrading retrieval quality. This module prevents "poisoning" vector databases with substandard material.

### Three Quality Pillars

1. **Chunk Quality (Intrinsic)** - Scores individual chunks 0-100 based on information density, semantic completeness, readability, metadata quality, and length
2. **Duplicate Detection (Relational)** - Uses MinHash/LSH for O(n) complexity duplicate detection with <5% false positive rate
3. **Data Drift (Temporal)** - Monitors statistical distribution changes using Kolmogorov-Smirnov test

### Learning Objectives

- Implement quality scoring with weighted dimensions
- Detect exact and near-duplicates efficiently
- Monitor data drift and generate alerts
- Integrate quality gates into production pipelines
- Recognize when NOT to use automated quality validation

## Setup

Import required modules and load example data.

In [ ]:
# Imports
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd

# Import our module functions
from l2_m3_dataquality_validation import (
    ChunkMetadata,
    ChunkQualityScorer,
    DuplicateDetector,
    DataDriftDetector,
    filter_low_quality_chunks,
    remove_duplicates
)

import config

# Configure logging for notebook
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

print("✓ Imports successful")

In [ ]:
# Load example data
with open('example_data.json', 'r') as f:
    example_data = json.load(f)

chunks_data = example_data['chunks']
baseline_metrics = example_data['baseline_metrics']

print(f"Loaded {len(chunks_data)} example chunks")
print(f"Baseline metrics: {len(baseline_metrics['quality_scores'])} samples")

# Expected: 10 chunks with various quality levels

## Section 1: Chunk Quality Scoring (Intrinsic)

Quality scoring evaluates chunks across **five weighted dimensions**:

1. **Information Density (30%)** - Penalizes boilerplate patterns and repetition
2. **Semantic Completeness (25%)** - Checks for complete sentences vs fragments
3. **Readability (20%)** - Validates sentence structure and encoding
4. **Metadata Quality (15%)** - Confirms required fields (source, date, section)
5. **Length Appropriateness (10%)** - Optimal range 200-800 characters

**Total Score:** 0-100, pass threshold typically 70

### 1.1: Score a Single Chunk

In [ ]:
# Initialize quality scorer
scorer = ChunkQualityScorer(min_score=70.0)

# Score a high-quality chunk (chunk_001 from example data)
chunk = chunks_data[0]
text = chunk['text']
metadata = ChunkMetadata(
    source=chunk['metadata']['source'],
    date=chunk['metadata']['date'],
    section=chunk['metadata']['section']
)

score = scorer.score_chunk(text, metadata)

print(f"Chunk ID: {chunk['chunk_id']}")
print(f"Expected Quality: {chunk['expected_quality']}")
print(f"\\nScoring Results:")
print(f"  Total Score: {score.total_score}")
print(f"  Passed: {score.passed}")
print(f"  Breakdown:")
print(f"    - Information Density: {score.information_density}")
print(f"    - Semantic Completeness: {score.semantic_completeness}")
print(f"    - Readability: {score.readability}")
print(f"    - Metadata Quality: {score.metadata_quality}")
print(f"    - Length Appropriateness: {score.length_appropriateness}")

# Expected: Score ~75-85, passed=True for high-quality chunk

## Summary & Key Takeaways

### Three Quality Pillars Mastered

1. **Chunk Quality (Intrinsic)** ✓
   - 5 weighted dimensions: density, completeness, readability, metadata, length
   - Pass threshold: 70-85 typically
   - Reject worst 10-20% of chunks

2. **Duplicate Detection (Relational)** ✓
   - MinHash LSH: O(n) vs O(n²) complexity
   - Standard threshold: 0.85 for near-duplicates
   - ~2-5% false negative rate acceptable

3. **Data Drift (Temporal)** ✓
   - Kolmogorov-Smirnov statistical test
   - Alert on >15% distribution shift
   - Actionable recommendations for degradation

### Production Metrics to Monitor

- **Quality Pass Rate:** Target 70-85%, alert if <60%
- **Deduplication Rate:** Target 5-20%, alert if >25% or 0%
- **Drift Score:** Target <0.20, alert if >0.25
- **Pipeline Duration:** Target <30 min, alert if >45 min

### When NOT to Use

- Early MVP (<500 docs) → Manual review
- Real-time (<5s latency) → Sampling validation
- Curated sources (<1% bad data) → Skip quality validation
- Multi-tenant → Managed platform
- Cost-constrained → Async validation

### Next Steps

1. **Calibrate thresholds** on your real data (1-2 weeks)
2. **Set up monitoring** (Prometheus + Grafana)
3. **Integrate into pipeline** (Airflow/custom orchestration)
4. **Monitor and tune** (2-4 hours/week ongoing)

### Resources

- `l2_m3_dataquality_validation.py` - Core implementation
- `app.py` - FastAPI wrapper for production
- `tests_smoke.py` - Validation tests
- `README.md` - Full documentation

**Module Complete!** Proceed to M5.4: Monitoring & Observability

In [ ]:
# Full validation pipeline combining all components
print("=== FULL VALIDATION PIPELINE ===\n")

# Step 1: Prepare chunks
all_chunks = [
    (chunk['chunk_id'], chunk['text'], 
     ChunkMetadata(**chunk['metadata']) if chunk['metadata'] else None)
    for chunk in chunks_data
]

print(f"Step 1: Input chunks: {len(all_chunks)}")

# Step 2: Quality filtering
passed_chunks, quality_scores = filter_low_quality_chunks(all_chunks, min_score=70.0)
quality_pass_rate = len(passed_chunks) / len(all_chunks) * 100
print(f"Step 2: After quality filter: {len(passed_chunks)} ({quality_pass_rate:.1f}% pass rate)")

# Step 3: Deduplication
unique_ids = remove_duplicates(passed_chunks, threshold=0.85)
dedup_rate = (len(passed_chunks) - len(unique_ids)) / len(passed_chunks) * 100
print(f"Step 3: After deduplication: {len(unique_ids)} ({dedup_rate:.1f}% dedup rate)")

# Step 4: Calculate final metrics for drift monitoring
final_quality_scores = [s.total_score for s in quality_scores if s.passed]
final_lengths = [len(text) for _, text in passed_chunks if _ in unique_ids]

print(f"\n✓ Final validated chunks: {len(unique_ids)} / {len(all_chunks)}")
print(f"✓ Overall pass rate: {len(unique_ids) / len(all_chunks) * 100:.1f}%")
print(f"✓ Avg quality score: {np.mean(final_quality_scores):.1f}")
print(f"✓ Avg chunk length: {np.mean(final_lengths) if final_lengths else 0:.0f} chars")

# Expected: ~5-6 chunks pass quality, ~4-5 unique after dedup

## Section 6: Production Integration

### Pipeline Flow

```
Documents → Chunking → Quality Filter → Deduplication → Drift Check → Vector DB
                          ↓               ↓                ↓
                       Reject <70      Remove >85%      Alert if >0.25
                                      similarity        drift score
```

### Full Validation Pipeline Example

### ✅ Use This Approach When:

- Processing 10K-500K chunks/day
- Diverse data sources (web scraping, PDFs, user uploads)
- Duplicate rate >10% historically
- Team size: 2+ engineers
- Storage/compute costs matter
- Retrieval quality degradation observed

### 💰 Cost Summary

| Scale | Chunks/Day | Cost/Month | Changes Needed |
|-------|------------|------------|----------------|
| Small | 10K | $15 | None |
| Medium | 100K | $80 | Multiprocessing, Prometheus rules |
| Large | 1M+ | $400-600 | Distributed dedup, streaming |

**Setup:** 2-3 days implementation + 1-2 weeks calibration  
**Maintenance:** 2-4 hours/week threshold tuning

## Section 5: Decision Card - When NOT to Use

Automated quality validation isn't universally optimal. Recognize when simpler approaches are better.

### ❌ Scenario 1: Early MVP (<500 documents, changing >30% weekly)

**Problem:** Calibration overhead exceeds benefits

**Use instead:** Manual review
- **Cost:** $500-2000/month human time
- **Benefit:** Catches semantic issues automation misses
- **Trade-off:** Doesn't scale past 5K chunks

### ❌ Scenario 2: Real-time ingestion (<5 second latency required)

**Problem:** Quality scoring adds 15-30% overhead

**Use instead:** Sampling validation (validate 10-15% only)
- **Cost:** $40/month (80% less compute)
- **Benefit:** Meets latency requirements
- **Trade-off:** 95% quality vs 99.9% quality, 1% slip-through rate

### ❌ Scenario 3: Curated sources (peer-reviewed, <1% bad data)

**Problem:** Automation costs exceed bad-data costs

**Use instead:** Skip quality validation; MinHash dedup only
- **Savings:** 50% less pipeline time
- **Risk:** Minimal with pre-vetted sources

### ❌ Scenario 4: Multi-tenant (different quality standards per customer)

**Problem:** Complex per-tenant calibration

**Use instead:** Managed platform with per-tenant config (Monte Carlo, Soda, Great Expectations Cloud)
- **Cost:** $500-2000/month
- **Benefit:** Handles multi-tenancy out-of-box
- **Trade-off:** Vendor lock-in

### ❌ Scenario 5: Cost-constrained (quality validation exceeds bad-data costs)

**Problem:** GPU pipeline budget tight

**Use instead:** Async validation on separate CPU-only pipeline
- **Savings:** Run validation overnight on cheaper instances
- **Benefit:** Decouple from critical path

### Failure 2: Miscalibrated Quality Thresholds

**Problem:** `min_score=90` rejects 70-80% of valid chunks (too strict).

**Fix:** Analyze real data distribution; set threshold at 10th-20th percentile; reject worst 10-20% only.

### Failure 3: Drift Detection Sensitivity

**Problem:** `significance_level=0.10`, `drift_threshold=0.05` fires false alarms on natural variation.

**Fix:**
- Use 0.05 significance level (standard)
- Threshold 0.15 (15% shift, not 5%)
- Require minimum 50 samples
- Implement 6-hour cooldown between alerts

### Failure 4: Dashboard Performance

**Problem:** Plotting 1M metrics over 30 days causes 30+ second timeouts.

**Fix:**
- Batch metric flushes (every 60s, not per-chunk)
- Use Prometheus recording rules
- Limit Grafana to <2000 data points (downsample)

### Failure 5: Quality Scoring Bottleneck

**Problem:** Serial processing 50K chunks takes 280 seconds (4.5 minutes).

**Fix:**
- Enable multiprocessing (8 workers = 8x speedup)
- Batch operations (process 1000 chunks at a time)
- Cache identical chunks (avoid re-scoring)

In [ ]:
# Reproduce Failure 1: Too strict threshold
text1 = "Complete training by Dec 31, 2024 to qualify for certification"
text2 = "Complete training by Dec 31, 2025 to qualify for certification"

# Bad: threshold 0.95 (too strict)
detector_bad = DuplicateDetector(threshold=0.95)
detector_bad.add_chunk("chunk1", text1)
dups_bad = detector_bad.find_duplicates("chunk2", text2)

# Good: threshold 0.85 (standard)
detector_good = DuplicateDetector(threshold=0.85)
detector_good.add_chunk("chunk1", text1)
dups_good = detector_good.find_duplicates("chunk2", text2)

print("❌ BAD (threshold=0.95):")
print(f"  Duplicates found: {len(dups_bad)}")
for dup_id, sim in dups_bad:
    print(f"    {dup_id}: similarity {sim:.3f} (FALSE POSITIVE - dates differ)")

print("\n✓ GOOD (threshold=0.85):")
print(f"  Duplicates found: {len(dups_good)}")
print("  (Correctly ignores minor date variation)")

# Expected: 0.95 flags as duplicate, 0.85 does not

## Section 4: Common Failures & Fixes

Learn from production failure modes to calibrate your quality gates correctly.

### Failure 1: False Positive Duplicates

**Problem:** High similarity threshold (0.95) flags "training by Dec 31, 2024" vs "2025" as duplicate.

**Symptoms:**
- Dedup rate suddenly >25%
- Alert: `dedup_rate_percent > 25.0`

**Fix:**
1. Use standard 0.85 threshold (not 0.95)
2. Add date-aware filtering if timestamps change frequently
3. Monitor dedup rate (alert if >25%)

In [ ]:
# Initialize drift detector
drift_detector = DataDriftDetector(significance_level=0.05, drift_threshold=0.15)

# Set baseline from example data
drift_detector.set_baseline(
    quality_scores=baseline_metrics['quality_scores'],
    chunk_lengths=baseline_metrics['chunk_lengths'],
    information_density=baseline_metrics['information_density']
)

# Simulate degraded current batch (lower quality)
current_quality = [60, 65, 62, 63, 61, 64, 62, 65, 63, 61] * 7  # 70 samples
current_lengths = [500, 510, 490, 505, 495, 515, 485, 500, 510, 495] * 7
current_density = [68, 70, 66, 69, 67, 71, 65, 68, 70, 66] * 7

# Detect drift
drift_result = drift_detector.detect_drift(
    quality_scores=current_quality,
    chunk_lengths=current_lengths,
    information_density=current_density
)

print(f"Drift Detected: {drift_result['drift_detected']}")
print(f"\nMetrics:")
for metric_name, metric_data in drift_result['metrics'].items():
    print(f"\n  {metric_name}:")
    print(f"    Baseline mean: {metric_data['baseline_mean']}")
    print(f"    Current mean: {metric_data['current_mean']}")
    print(f"    Shift: {metric_data['mean_shift']} ({metric_data['percent_shift']}%)")
    print(f"    Drift detected: {metric_data['drift_detected']}")

if drift_result.get('recommendations'):
    print(f"\n✓ Recommendations:")
    for rec in drift_result['recommendations']:
        print(f"  - {rec}")

# Expected: Drift detected with quality degradation recommendation

## Section 3: Data Drift Detection (Temporal)

Monitors statistical distribution shifts using **Kolmogorov-Smirnov (K-S) test**.

**How it works:**
1. Set baseline from historical metrics (quality scores, lengths, density)
2. Collect current batch metrics
3. Run K-S test for each metric independently
4. Flag drift if p-value < 0.05 AND K-S statistic > 0.15
5. Generate actionable recommendations

**Alert Conditions:**
- Quality degradation > 10 points
- Length shift > 30%
- Density drop > 15 points

### 3.1: Set Baseline and Detect Drift

In [ ]:
# Test different thresholds
thresholds = [0.95, 0.85, 0.70]

for threshold in thresholds:
    detector_test = DuplicateDetector(threshold=threshold)
    unique_ids_test, dup_info_test = detector_test.deduplicate_batch(test_chunks)
    dedup_rate = len(dup_info_test) / len(test_chunks) * 100
    print(f"Threshold {threshold}: {len(unique_ids_test)} unique, {len(dup_info_test)} duplicates ({dedup_rate:.0f}%)")

# Expected: Higher threshold = fewer duplicates detected

### 2.2: Threshold Impact

Different thresholds catch different types of duplicates:
- **0.95+**: Only exact or near-exact duplicates
- **0.85** (standard): Near-duplicates with minor variations
- **0.70**: More aggressive, may flag false positives

In [ ]:
# Initialize duplicate detector
detector = DuplicateDetector(threshold=0.85)

# Prepare chunks (use chunks 004, 005, 006 which are duplicates/near-duplicates)
test_chunks = [
    (chunks_data[3]['chunk_id'], chunks_data[3]['text']),  # chunk_004 - original
    (chunks_data[4]['chunk_id'], chunks_data[4]['text']),  # chunk_005 - exact duplicate
    (chunks_data[5]['chunk_id'], chunks_data[5]['text']),  # chunk_006 - near duplicate
    (chunks_data[7]['chunk_id'], chunks_data[7]['text']),  # chunk_008 - unique
]

# Run deduplication
unique_ids, dup_info = detector.deduplicate_batch(test_chunks)

print(f"Input: {len(test_chunks)} chunks")
print(f"Unique: {len(unique_ids)} chunks")
print(f"Duplicates: {len(dup_info)}")
print(f"\nUnique chunk IDs: {unique_ids}")
print(f"\nDuplicates detected:")
for dup_id, orig_id, similarity in dup_info:
    print(f"  {dup_id} is duplicate of {orig_id} (similarity: {similarity:.3f})")

# Expected: 2 unique (004, 008), 2 duplicates (005, 006)

## Section 2: Duplicate Detection (Relational)

MinHash LSH enables **O(n) complexity** duplicate detection vs O(n²) pairwise comparison.

**Key Features:**
- Jaccard similarity on tokenized text
- Configurable threshold (0.85 standard for near-duplicates)
- Probabilistic: ~2-5% false negative rate acceptable for RAG
- Scales to millions of chunks

**Trade-off:** LSH is probabilistic—may miss ~2-5% of duplicates. For RAG, this is acceptable; perfect recall isn't worth 100x slowdown.

### 2.1: Detect Exact Duplicates

In [ ]:
# Prepare chunks for batch scoring
chunks_for_scoring = []
for chunk in chunks_data:
    metadata = None
    if chunk['metadata']:
        metadata = ChunkMetadata(
            source=chunk['metadata']['source'],
            date=chunk['metadata']['date'],
            section=chunk['metadata']['section']
        )
    chunks_for_scoring.append((chunk['text'], metadata))

# Batch score
scores = scorer.batch_score(chunks_for_scoring)

# Create results dataframe
results_df = pd.DataFrame([
    {
        'chunk_id': chunks_data[i]['chunk_id'],
        'expected_quality': chunks_data[i]['expected_quality'],
        'total_score': s.total_score,
        'passed': s.passed,
        'info_density': s.information_density,
        'completeness': s.semantic_completeness,
        'readability': s.readability
    }
    for i, s in enumerate(scores)
])

print(results_df.to_string(index=False))
print(f"\n✓ Pass Rate: {sum(s.passed for s in scores) / len(scores) * 100:.1f}%")

# Expected: ~5-6 chunks pass, 4-5 fail

### 1.3: Batch Quality Scoring

Score all chunks and analyze the distribution of quality scores.

In [ ]:
# Score a low-quality chunk (chunk_002 - high boilerplate)
low_quality_chunk = chunks_data[1]

score_low = scorer.score_chunk(low_quality_chunk['text'], None)

print("=== LOW-QUALITY CHUNK ===")
print(f"Chunk ID: {low_quality_chunk['chunk_id']}")
print(f"Text: {low_quality_chunk['text'][:80]}...")
print(f"Total Score: {score_low.total_score}")
print(f"Passed: {score_low.passed}")
print(f"Failure Reasons: {score_low.failure_reasons}")

print("\n=== HIGH-QUALITY CHUNK ===")
print(f"Chunk ID: {chunk['chunk_id']}")
print(f"Total Score: {score.total_score}")
print(f"Passed: {score.passed}")

# Expected: Low score <40, high score >75

### 1.2: Compare Low-Quality vs High-Quality